# DataGate — SynapseML LightGBM HPO từ Iceberg Catalog

Notebook này dùng để huấn luyện thử nghiệm trước khi đưa bộ siêu tham số vào job vận hành phát hiện bất thường.

Luồng chính:

1. Đọc dữ liệu trực tiếp từ Iceberg catalog bằng Spark, không đọc qua Trino.
2. Chọn `processing_date_hour` làm target batch (`label = 1`).
3. Lấy batch liền trước và các batch lịch sử làm dữ liệu so sánh (`label = 0`).
4. Train SynapseML LightGBM và dùng Optuna để tìm hyperparameter tốt nhất.
5. Xuất file JSON đúng template để nạp vào `model_parameters` hoặc dùng cho job vận hành.

## 1. Import libraries

In [1]:
import os
import gc
import json
import time
import warnings
from pathlib import Path

import optuna
import pandas as pd

from optuna.pruners import NopPruner
from optuna.samplers import TPESampler

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    ByteType,
    ShortType,
    IntegerType,
    LongType,
    FloatType,
    DoubleType,
    DecimalType,
)

from pyspark.storagelevel import StorageLevel
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler

from synapse.ml.lightgbm import LightGBMClassifier

warnings.filterwarnings("ignore")

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Cấu hình notebook

Chỉ cần sửa cell này khi đổi bảng, đổi batch hoặc đổi cấu hình cluster.

In [ ]:
# ============================================================
# Reproducibility / output
# ============================================================
RANDOM_STATE = 42

OUTPUT_DIR = "/opt/spark/workspace/output/datagate_lgbm_hpo"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

HPO_OUTPUT_JSON = f"{OUTPUT_DIR}/lgbm_hpo_params.json"
HPO_SUMMARY_JSON = f"{OUTPUT_DIR}/lgbm_hpo_summary.json"
OPTUNA_TRIALS_CSV = f"{OUTPUT_DIR}/optuna_trials.csv"

# ============================================================
# Iceberg table
# ============================================================
CATALOG_NAME = "iceberg"
SCHEMA_NAME = "silver"
TABLE_NAME = "yellow_tripdata"

BATCH_TIME_COL = "processing_date_hour"

# Chọn batch target:
# - Nếu TARGET_PROCESSING_DATE_HOUR có giá trị, notebook dùng đúng batch đó.
# - Nếu để None, notebook tự chọn batch mới nhất trong BATCH_DAY.
# - Nếu cả hai đều None, notebook chọn batch mới nhất toàn bảng.
BATCH_DAY = "2025-01-15"
TARGET_PROCESSING_DATE_HOUR = "2025-01-15 12:00:00"

# Giống logic vận hành: target là batch hiện tại, history là batch so sánh.
PREVIOUS_BATCH_HOURS = 12
REQUIRED_HISTORY_DAYS = 7
HISTORY_DAYS = [1, 7, 14]

# ============================================================
# Sampling / split
# ============================================================
TARGET_SAMPLE_PER_GROUP = 10_000
MIN_ROWS_PER_GROUP = 100

VALID_SIZE = 0.15
TEST_SIZE = 0.15

# ============================================================
# Feature config
# ============================================================
# exclude_cols nên khớp anomaly_configs.exclude_cols trong DB nếu có.
EXCLUDE_COLS = [
    "label",
    BATCH_TIME_COL,
    "date_hour",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
]

# Các cột categorical sẽ được StringIndexer trước khi đưa vào LightGBM.
CATEGORICAL_COLS = [
    "vendorid",
    "ratecodeid",
    "store_and_fwd_flag",
    "payment_type",
]

# Nếu để rỗng, notebook tự lấy các cột numeric còn lại.
NUMERIC_COLS = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "pulocationid",
    "dolocationid",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
    "cbd_congestion_fee",
]

# ============================================================
# Spark / SynapseML cluster profile
# ============================================================
SPARK_MASTER = os.getenv("SPARK_MASTER", "spark://spark-master:7077")

LGBM_NUM_TASKS = 4
LGBM_NUM_THREADS = 3
SPARK_SQL_SHUFFLE_PARTITIONS = 12
SPARK_DEFAULT_PARALLELISM = 12

# ============================================================
# Iceberg REST catalog config
# ============================================================
CATALOG_REST_URI = os.getenv("ICEBERG_REST_URI", "http://iceberg-rest:8181")
CATALOG_WAREHOUSE = os.getenv("ICEBERG_WAREHOUSE", "s3://warehouse")

S3_ENDPOINT = os.getenv("AWS_S3_ENDPOINT", "http://minio:9000")
S3_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID", "admin")
S3_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "password")
S3_REGION = os.getenv("AWS_REGION", "us-east-1")

# ============================================================
# Optuna / LightGBM
# ============================================================
RUN_OPTUNA = True
N_TRIALS = 50
DEBUG_TRIAL_ERRORS = False

N_ESTIMATORS_MAX = 2000
EARLY_STOPPING_ROUNDS = 100

print("Output JSON:", HPO_OUTPUT_JSON)
print("Table:", f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}")
print("Target processing_date_hour:", TARGET_PROCESSING_DATE_HOUR)
print("Batch day:", BATCH_DAY)
print("History days:", HISTORY_DAYS)
print("Optuna:", {"run": RUN_OPTUNA, "n_trials": N_TRIALS})

## 3. Tạo Spark session đọc trực tiếp Iceberg catalog

In [ ]:
def validate_name(value, field_name):
    value = str(value).strip()
    if not value:
        raise ValueError(f"{field_name} must not be empty.")
    for char in value:
        if not (char.isalnum() or char == "_"):
            raise ValueError(f"Invalid {field_name}: {value}")
    return value


CATALOG_NAME = validate_name(CATALOG_NAME, "CATALOG_NAME")
SCHEMA_NAME = validate_name(SCHEMA_NAME, "SCHEMA_NAME")
TABLE_NAME = validate_name(TABLE_NAME, "TABLE_NAME")
BATCH_TIME_COL = validate_name(BATCH_TIME_COL, "BATCH_TIME_COL")

FULL_TABLE_NAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"
print("FULL_TABLE_NAME:", FULL_TABLE_NAME)

In [ ]:
spark = (
    SparkSession.builder
    .appName("DataGate SynapseML LightGBM HPO")
    .master(SPARK_MASTER)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", CATALOG_REST_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", CATALOG_WAREHOUSE)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.endpoint", S3_ENDPOINT)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.access-key-id", S3_ACCESS_KEY)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.secret-access-key", S3_SECRET_KEY)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.path-style-access", "true")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.region", S3_REGION)
    .config("spark.sql.shuffle.partitions", str(SPARK_SQL_SHUFFLE_PARTITIONS))
    .config("spark.default.parallelism", str(SPARK_DEFAULT_PARALLELISM))
    .config("spark.dynamicAllocation.enabled", "false")
    .config("spark.task.cpus", str(LGBM_NUM_THREADS))
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark app:", spark.sparkContext.appName)
print("Spark master:", spark.sparkContext.master)
print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

## 4. Đọc bảng và chọn `processing_date_hour`

In [ ]:
df_table = spark.table(FULL_TABLE_NAME)

if BATCH_TIME_COL not in df_table.columns:
    raise ValueError(f"Missing batch time column: {BATCH_TIME_COL}")

df_table = df_table.withColumn(BATCH_TIME_COL, F.col(BATCH_TIME_COL).cast("timestamp"))

df_table.select(
    F.count("*").alias("row_count"),
    F.min(BATCH_TIME_COL).alias("min_processing_date_hour"),
    F.max(BATCH_TIME_COL).alias("max_processing_date_hour"),
).show(truncate=False)

df_table.printSchema()

In [ ]:
def ts_str(value):
    return pd.Timestamp(value).strftime("%Y-%m-%d %H:%M:%S")


def day_range(day):
    start = pd.Timestamp(day)
    end = start + pd.Timedelta(days=1)
    return ts_str(start), ts_str(end)


def list_batch_hours(source_df, batch_day=None, limit=50):
    data = source_df

    if batch_day:
        start, end = day_range(batch_day)
        data = data.filter(
            (F.col(BATCH_TIME_COL) >= F.to_timestamp(F.lit(start))) &
            (F.col(BATCH_TIME_COL) < F.to_timestamp(F.lit(end)))
        )

    return (
        data
        .select(BATCH_TIME_COL)
        .distinct()
        .orderBy(F.col(BATCH_TIME_COL).desc())
        .limit(int(limit))
    )


print("Available processing_date_hour:")
list_batch_hours(df_table, BATCH_DAY).show(50, truncate=False)

In [ ]:
def select_target_hour(source_df):
    if TARGET_PROCESSING_DATE_HOUR:
        return pd.Timestamp(TARGET_PROCESSING_DATE_HOUR)

    data = source_df

    if BATCH_DAY:
        start, end = day_range(BATCH_DAY)
        data = data.filter(
            (F.col(BATCH_TIME_COL) >= F.to_timestamp(F.lit(start))) &
            (F.col(BATCH_TIME_COL) < F.to_timestamp(F.lit(end)))
        )

    row = data.select(F.max(BATCH_TIME_COL).alias("target_hour")).first()

    if row is None or row["target_hour"] is None:
        raise ValueError("Cannot select target processing_date_hour.")

    return pd.Timestamp(row["target_hour"])


target_hour = select_target_hour(df_table)

print("Selected target_hour:", ts_str(target_hour))

## 5. Tạo target batch và history batches

In [ ]:
def comparison_hours(target):
    target = pd.Timestamp(target)

    hours = [target - pd.Timedelta(hours=PREVIOUS_BATCH_HOURS)]
    hours += [target - pd.Timedelta(days=int(day)) for day in HISTORY_DAYS]

    return sorted(set(hours))


def batch_count(source_df, hour):
    return source_df.filter(
        F.col(BATCH_TIME_COL) == F.to_timestamp(F.lit(ts_str(hour)))
    ).count()


history_hours = comparison_hours(target_hour)
required_history_hour = target_hour - pd.Timedelta(days=REQUIRED_HISTORY_DAYS)

print("Target:", ts_str(target_hour), "| rows:", batch_count(df_table, target_hour))
print("Required history:", ts_str(required_history_hour), "| rows:", batch_count(df_table, required_history_hour))

print("\\nComparison hours:")
for hour in history_hours:
    print("-", ts_str(hour), "| rows:", batch_count(df_table, hour))

In [ ]:
if batch_count(df_table, target_hour) == 0:
    raise ValueError(f"Target batch not found: {ts_str(target_hour)}")

if batch_count(df_table, required_history_hour) == 0:
    raise ValueError(f"Required history batch not found: {ts_str(required_history_hour)}")

needed_hours = [target_hour] + history_hours
needed_hours_sql = ", ".join([f"TIMESTAMP '{ts_str(hour)}'" for hour in needed_hours])

df_needed = spark.sql(f"""
    SELECT *
    FROM {FULL_TABLE_NAME}
    WHERE {BATCH_TIME_COL} IN ({needed_hours_sql})
""")

df_needed = df_needed.withColumn(BATCH_TIME_COL, F.col(BATCH_TIME_COL).cast("timestamp"))
df_needed = df_needed.persist(StorageLevel.MEMORY_AND_DISK)

print("Loaded rows for selected batches:", df_needed.count())
df_needed.groupBy(BATCH_TIME_COL).count().orderBy(BATCH_TIME_COL).show(truncate=False)

In [ ]:
def filter_batch(source_df, hour):
    return source_df.filter(
        F.col(BATCH_TIME_COL) == F.to_timestamp(F.lit(ts_str(hour)))
    )


def sample_batch(source_df, hour, label, max_rows, seed):
    data = filter_batch(source_df, hour)
    data = data.withColumn("label", F.lit(float(label)))

    row_count = data.count()

    if row_count <= int(max_rows):
        return data

    return data.orderBy(F.rand(int(seed))).limit(int(max_rows))


def union_many(parts):
    result = parts[0]

    for part in parts[1:]:
        result = result.unionByName(part, allowMissingColumns=True)

    return result

In [ ]:
positive_df = sample_batch(
    df_needed,
    target_hour,
    label=1,
    max_rows=TARGET_SAMPLE_PER_GROUP,
    seed=RANDOM_STATE,
)

negative_parts = []

for index, hour in enumerate(history_hours):
    part = sample_batch(
        df_needed,
        hour,
        label=0,
        max_rows=TARGET_SAMPLE_PER_GROUP,
        seed=RANDOM_STATE + index + 1,
    )

    rows = part.count()

    if rows < MIN_ROWS_PER_GROUP:
        print(f"Skip history batch {ts_str(hour)} because rows={rows}")
        continue

    negative_parts.append(part)
    print(f"Use history batch {ts_str(hour)} | rows={rows}")

if positive_df.count() < MIN_ROWS_PER_GROUP:
    raise ValueError("Target batch does not have enough rows.")

if not negative_parts:
    raise ValueError("No valid history batch for label=0.")

work_df = union_many([positive_df] + negative_parts)
work_df = work_df.repartition(SPARK_SQL_SHUFFLE_PARTITIONS)
work_df = work_df.persist(StorageLevel.MEMORY_AND_DISK)

print("Training dataset rows:", work_df.count())
work_df.groupBy("label").count().orderBy("label").show()
work_df.groupBy(BATCH_TIME_COL, "label").count().orderBy(BATCH_TIME_COL, "label").show(truncate=False)

## 6. Chọn feature columns

In [ ]:
NUMERIC_TYPES = (
    ByteType,
    ShortType,
    IntegerType,
    LongType,
    FloatType,
    DoubleType,
    DecimalType,
)

schema_map = {field.name: field.dataType for field in work_df.schema.fields}

exclude_cols = set(EXCLUDE_COLS)
exclude_cols.add("label")
exclude_cols.add(BATCH_TIME_COL)

categorical_cols = [
    col for col in CATEGORICAL_COLS
    if col in work_df.columns and col not in exclude_cols
]

if NUMERIC_COLS:
    numeric_cols = [
        col for col in NUMERIC_COLS
        if col in work_df.columns and col not in exclude_cols and col not in categorical_cols
    ]
else:
    numeric_cols = [
        name for name, dtype in schema_map.items()
        if name not in exclude_cols
        and name not in categorical_cols
        and isinstance(dtype, NUMERIC_TYPES)
    ]

feature_input_cols = numeric_cols + categorical_cols

if not feature_input_cols:
    raise ValueError("No usable feature columns found.")

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)
print("Total feature inputs:", len(feature_input_cols))

## 7. Split train / validation / test

In [ ]:
def split_work_df(source_df):
    data = source_df.withColumn("_rnd", F.rand(RANDOM_STATE))

    test = data.filter(F.col("_rnd") < TEST_SIZE).drop("_rnd")
    valid = data.filter(
        (F.col("_rnd") >= TEST_SIZE) &
        (F.col("_rnd") < TEST_SIZE + VALID_SIZE)
    ).drop("_rnd")
    train = data.filter(F.col("_rnd") >= TEST_SIZE + VALID_SIZE).drop("_rnd")

    return train, valid, test


def check_split(name, split_df):
    rows = split_df.count()
    labels = split_df.select("label").distinct().count()

    print(name, "| rows:", rows, "| labels:", labels)

    if rows == 0:
        raise ValueError(f"{name} split is empty.")

    if labels < 2:
        raise ValueError(f"{name} split does not contain both labels.")


train_df, valid_df, test_df = split_work_df(work_df)

check_split("train", train_df)
check_split("valid", valid_df)
check_split("test", test_df)

## 8. Build feature vector cho SynapseML LightGBM

In [ ]:
def fit_indexers(train_source_df, cat_cols):
    models = []
    data = train_source_df

    for col in cat_cols:
        idx_col = f"{col}__idx"
        data = data.withColumn(col, F.col(col).cast("string"))

        model = StringIndexer(
            inputCol=col,
            outputCol=idx_col,
            handleInvalid="keep",
        ).fit(data)

        models.append(model)
        data = model.transform(data)

    return models


def transform_features(source_df, indexer_models, num_cols, cat_cols):
    data = source_df

    for col in num_cols:
        data = data.withColumn(col, F.col(col).cast("double"))

    for col in cat_cols:
        data = data.withColumn(col, F.col(col).cast("string"))

    for model in indexer_models:
        data = model.transform(data)

    indexed_cols = [f"{col}__idx" for col in cat_cols]
    final_feature_cols = num_cols + indexed_cols

    assembler = VectorAssembler(
        inputCols=final_feature_cols,
        outputCol="features",
        handleInvalid="keep",
    )

    return assembler.transform(data).select(
        "features",
        F.col("label").cast("double").alias("label"),
    )

In [ ]:
indexer_models = fit_indexers(train_df, categorical_cols)

train_features_df = transform_features(train_df, indexer_models, numeric_cols, categorical_cols)
valid_features_df = transform_features(valid_df, indexer_models, numeric_cols, categorical_cols)
test_features_df = transform_features(test_df, indexer_models, numeric_cols, categorical_cols)

train_features_df = train_features_df.persist(StorageLevel.MEMORY_AND_DISK)
valid_features_df = valid_features_df.persist(StorageLevel.MEMORY_AND_DISK)
test_features_df = test_features_df.persist(StorageLevel.MEMORY_AND_DISK)

print("Feature rows:")
print("train:", train_features_df.count())
print("valid:", valid_features_df.count())
print("test :", test_features_df.count())

feature_cols = numeric_cols + categorical_cols
categorical_slot_indexes = list(range(len(numeric_cols), len(numeric_cols) + len(categorical_cols)))

print("Feature cols:", feature_cols)
print("Categorical slot indexes:", categorical_slot_indexes)

In [ ]:
def with_validation_flag(features_df, is_validation):
    return (
        features_df
        .select("features", "label")
        .withColumn("is_validation", F.lit(bool(is_validation)))
    )


lgbm_train_valid_df = with_validation_flag(train_features_df, False)
lgbm_train_valid_df = lgbm_train_valid_df.unionByName(
    with_validation_flag(valid_features_df, True)
)

lgbm_train_valid_df = lgbm_train_valid_df.repartition(LGBM_NUM_TASKS)
lgbm_train_valid_df = lgbm_train_valid_df.persist(StorageLevel.MEMORY_AND_DISK)

print("LightGBM train+valid rows:", lgbm_train_valid_df.count())
lgbm_train_valid_df.groupBy("is_validation", "label").count().orderBy("is_validation", "label").show()

## 9. LightGBM helper functions

In [ ]:
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)

BASELINE_PARAMS = {
    "learningRate": 0.05,
    "numLeaves": 31,
    "maxDepth": -1,
    "minDataInLeaf": 20,
    "baggingFraction": 1.0,
    "baggingFreq": 0,
    "featureFraction": 1.0,
    "lambdaL1": 1e-8,
    "lambdaL2": 1e-8,
    "minGainToSplit": 0.0,
    "maxBin": 127,
}


def build_full_lgbm_params(params):
    full_params = {
        "objective": "binary",
        "metric": "auc",
        "featuresCol": "features",
        "labelCol": "label",
        "validationIndicatorCol": "is_validation",
        "numIterations": N_ESTIMATORS_MAX,
        "earlyStoppingRound": EARLY_STOPPING_ROUNDS,
        "numTasks": LGBM_NUM_TASKS,
        "numThreads": LGBM_NUM_THREADS,
        "maxStreamingOMPThreads": LGBM_NUM_THREADS,
        "timeout": 600,
        "useBarrierExecutionMode": True,
        "useSingleDatasetMode": True,
        "dataTransferMode": "streaming",
        "verbosity": -1,
    }

    full_params.update(BASELINE_PARAMS)
    full_params.update(params)

    if categorical_slot_indexes:
        full_params["categoricalSlotIndexes"] = [int(x) for x in categorical_slot_indexes]

    return full_params


def build_classifier(params):
    return LightGBMClassifier(**build_full_lgbm_params(params))

In [ ]:
def safe_model_call(model, method_name):
    try:
        method = getattr(model, method_name, None)
        if method is None:
            return None
        return method() if callable(method) else method
    except Exception:
        return None


def to_optional_int(value):
    try:
        if value is None:
            return None
        return int(value)
    except Exception:
        return None


def get_iteration_info(model):
    raw_best = None
    raw_method = None

    for method_name in ["getBoosterBestIteration", "getBestIteration", "bestIteration"]:
        value = to_optional_int(safe_model_call(model, method_name))
        if value is not None:
            raw_best = value
            raw_method = method_name
            break

    total_iterations = to_optional_int(
        safe_model_call(model, "getBoosterNumTotalIterations")
    )

    if raw_best and raw_best > 0:
        best_iteration = raw_best
        source = raw_method
    else:
        best_iteration = total_iterations
        source = "fallback_total_iterations"

    return {
        "best_iteration": best_iteration,
        "best_iteration_source": source,
        "booster_total_iterations": total_iterations,
    }

In [ ]:
def evaluate_auc(model, data_df):
    pred_df = model.transform(data_df)
    return float(auc_evaluator.evaluate(pred_df))


def train_once(params):
    start_time = time.time()

    model = build_classifier(params).fit(lgbm_train_valid_df)

    valid_auc = evaluate_auc(model, valid_features_df)
    test_auc = evaluate_auc(model, test_features_df)

    result = {
        "valid_auc": valid_auc,
        "test_auc": test_auc,
        "fit_seconds": time.time() - start_time,
    }

    result.update(get_iteration_info(model))

    return result, model

## 10. Chạy baseline

In [ ]:
baseline_result, baseline_model = train_once(BASELINE_PARAMS)

print("Baseline result:")
print(json.dumps(baseline_result, indent=2, default=str))

del baseline_model
gc.collect()

## 11. Chạy Optuna HPO

In [ ]:
def suggest_lgbm_params(trial):
    return {
        "learningRate": trial.suggest_float("learningRate", 0.01, 0.20, log=True),
        "numLeaves": trial.suggest_categorical("numLeaves", [15, 31, 63, 127]),
        "maxDepth": trial.suggest_categorical("maxDepth", [-1, 4, 6, 8, 10]),
        "minDataInLeaf": trial.suggest_categorical("minDataInLeaf", [20, 50, 100, 200]),
        "baggingFraction": trial.suggest_float("baggingFraction", 0.70, 1.0),
        "baggingFreq": trial.suggest_categorical("baggingFreq", [0, 1, 3, 5]),
        "featureFraction": trial.suggest_float("featureFraction", 0.70, 1.0),
        "lambdaL1": trial.suggest_float("lambdaL1", 1e-8, 10.0, log=True),
        "lambdaL2": trial.suggest_float("lambdaL2", 1e-8, 10.0, log=True),
        "minGainToSplit": trial.suggest_float("minGainToSplit", 0.0, 1.0),
        "maxBin": trial.suggest_categorical("maxBin", [63, 127, 255]),
    }


def objective(trial):
    params = suggest_lgbm_params(trial)

    try:
        result, model = train_once(params)

        trial.set_user_attr("valid_auc", result["valid_auc"])
        trial.set_user_attr("test_auc", result["test_auc"])
        trial.set_user_attr("best_iteration", result["best_iteration"])
        trial.set_user_attr("fit_seconds", result["fit_seconds"])

        return float(result["valid_auc"])

    except Exception as exc:
        print(f"Trial {trial.number} failed:", repr(exc))

        if DEBUG_TRIAL_ERRORS:
            raise

        return 0.5

    finally:
        try:
            del model
        except Exception:
            pass

        gc.collect()

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=RANDOM_STATE),
    pruner=NopPruner(),
)

study.enqueue_trial(BASELINE_PARAMS)

study.optimize(
    objective,
    n_trials=N_TRIALS if RUN_OPTUNA else 1,
    show_progress_bar=True,
    gc_after_trial=True,
)

trials_df = study.trials_dataframe()
trials_df.to_csv(OPTUNA_TRIALS_CSV, index=False)

print("Best validation AUC:", study.best_value)
print("Best params:")
print(json.dumps(study.best_params, indent=2, default=str))
print("Saved trials:", OPTUNA_TRIALS_CSV)

## 12. Chọn tham số tốt nhất và train final model

In [ ]:
if float(study.best_value) >= float(baseline_result["valid_auc"]):
    selected_source = "optuna"
    selected_params = dict(study.best_params)
else:
    selected_source = "baseline"
    selected_params = dict(BASELINE_PARAMS)

print("Selected source:", selected_source)
print(json.dumps(selected_params, indent=2, default=str))

In [ ]:
final_result, final_model = train_once(selected_params)

print("Final result:")
print(json.dumps(final_result, indent=2, default=str))

## 13. Xuất JSON hyperparameter đúng template vận hành

In [ ]:
def build_production_json(selected_params, final_result):
    params = build_full_lgbm_params(selected_params)

    best_iteration = final_result.get("best_iteration")
    if best_iteration is None or int(best_iteration) <= 0:
        best_iteration = params["numIterations"]

    return {
        "learning_rate": float(params["learningRate"]),
        "num_leaves": int(params["numLeaves"]),
        "max_depth": int(params["maxDepth"]),
        "min_data_in_leaf": int(params["minDataInLeaf"]),
        "bagging_fraction": float(params["baggingFraction"]),
        "bagging_freq": int(params["baggingFreq"]),
        "feature_fraction": float(params["featureFraction"]),
        "lambda_l1": float(params["lambdaL1"]),
        "lambda_l2": float(params["lambdaL2"]),
        "min_gain_to_split": float(params["minGainToSplit"]),
        "max_bin": int(params["maxBin"]),
        "num_iterations": int(best_iteration),
    }


production_params = build_production_json(selected_params, final_result)

with open(HPO_OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(production_params, f, indent=4)

print("Saved production params:", HPO_OUTPUT_JSON)
print(json.dumps(production_params, indent=4))

In [ ]:
summary = {
    "table": FULL_TABLE_NAME,
    "batch_time_col": BATCH_TIME_COL,
    "target_processing_date_hour": ts_str(target_hour),
    "history_processing_date_hours": [ts_str(hour) for hour in history_hours],
    "selected_source": selected_source,
    "baseline_result": baseline_result,
    "final_result": final_result,
    "feature_cols": feature_cols,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "production_params_path": HPO_OUTPUT_JSON,
    "optuna_trials_path": OPTUNA_TRIALS_CSV,
}

with open(HPO_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4, default=str)

print("Saved summary:", HPO_SUMMARY_JSON)

## 14. Kiểm tra nhanh output

In [ ]:
expected_keys = {
    "learning_rate",
    "num_leaves",
    "max_depth",
    "min_data_in_leaf",
    "bagging_fraction",
    "bagging_freq",
    "feature_fraction",
    "lambda_l1",
    "lambda_l2",
    "min_gain_to_split",
    "max_bin",
    "num_iterations",
}

actual_keys = set(production_params.keys())

if actual_keys != expected_keys:
    raise ValueError(f"Invalid JSON keys: {sorted(actual_keys)}")

if production_params["num_iterations"] <= 0:
    raise ValueError("num_iterations must be greater than 0.")

if not (0 < production_params["learning_rate"] < 1):
    raise ValueError("learning_rate looks invalid.")

print("Output JSON format is valid.")

## 15. Cleanup

In [ ]:
for name in [
    "lgbm_train_valid_df",
    "train_features_df",
    "valid_features_df",
    "test_features_df",
    "work_df",
    "df_needed",
]:
    obj = globals().get(name)

    if obj is not None:
        try:
            obj.unpersist()
            print("Unpersisted:", name)
        except Exception as exc:
            print("Skip cleanup:", name, repr(exc))

gc.collect()
print("Cleanup completed.")